<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

## Why a Multiplier Measures Sensitivity

**Prove that the multiplier measures the change in the optimized value.**

Keep the clock data and objective; vary the calibration target and reoptimize both settings.

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 1 · Define the best attainable value

The fixed errors $d_i$ are $-2,-1,-1,0$ min at times $t_i=0,1,2,3$ h, respectively, for $i=1,\ldots,4$. Choose initial correction $q$ (min) and rate correction $r$ (min/h). The remaining errors and squared-error objective are

$$
y_i=d_i+q+rt_i,\qquad \phi(q,r)=\sum_{i=1}^4 y_i^2.
$$

Equations below use numerical values in these stated units.

Let $\beta$ (min) be the required correction at one hour. There is no correction bound. Write $x=(q,r)$, $c(x)=q+r$, and $h(x,\beta)=c(x)-\beta=0$. Only $\beta=1$ makes the one-hour reading exact. Define

$$
V(\beta)=\min_{c(x)=\beta}\phi(x),\qquad
L(x,\nu;\beta)=\phi(x)+\nu[c(x)-\beta].
$$

$V$ compares newly optimized decisions, not one fixed decision under changing requirements.

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 2 · Differentiate the constraint and objective

Assume a differentiable optimal solution $x^\star(\beta)$ near the target, a regular equality, and no direct dependence of $\phi$ on $\beta$. A prime denotes differentiation with respect to $\beta$. Differentiating feasibility gives

$$
c(x^\star(\beta))=\beta
\quad\Longrightarrow\quad
\nabla c(x^\star)^{\mathsf T}x^{\star\prime}(\beta)=1.
$$

Stationarity gives $\nabla\phi(x^\star)=-\nu^\star\nabla c(x^\star)$. Applying the chain rule to $V(\beta)=\phi(x^\star(\beta))$ therefore gives

$$
\begin{aligned}
V'(\beta)
&=\nabla\phi(x^\star)^{\mathsf T}x^{\star\prime}(\beta)\\
&=-\nu^\star\nabla c(x^\star)^{\mathsf T}x^{\star\prime}(\beta)\\
&=-\nu^\star.
\end{aligned}
$$

**The multiplier is the negative slope of the optimized value for this residual convention.** Thus $\Delta V\approx-\nu^\star\Delta\beta$. Writing the residual as $\beta-c(x)$ reverses the multiplier sign, while the physical value change is the same. Smoothness is needed for this derivative argument; it does not assert a derivative at a nonsmooth solution change.

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 3 · Verify the sensitivity exactly

The clock stationarity equations give

$$
q^\star=\frac{8\beta+1}{6},\qquad
r^\star=-\frac{2\beta+1}{6},\qquad
\nu^\star=\frac{26-20\beta}{3}.
$$

Substitution into the objective gives

$$
V(\beta)=0.2+\frac{10}{3}(\beta-1.3)^2,
\qquad V'(\beta)=\frac{20\beta-26}{3}=-\nu^\star.
$$

At $\beta=1$, $V=0.5$ min² and $\nu^\star=2$ min²/min. Raising the target by $0.05$ min predicts $\Delta V\approx-0.10$ min²; the exact change is $-0.09167$ min². The difference is the quadratic term $\tfrac{10}{3}(0.05)^2$.

The tangent gives the local estimate. At $\beta=1.3$, the equality passes through the unrestricted minimum, so the slope and multiplier are zero.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/03-2_constrained_optimization_lagrange_multipliers/assets/sensitivity_compact.svg" alt="The optimized value curve has slope minus 2 at target 1 and zero slope at target 1.3." width="512" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 4 · Interactive comparison

Only $\beta$ changes. The orange point is the newly optimized value, and the dashed tangent has slope $-\nu^\star$. The readout compares the exact value change from $\beta=1$ with the fixed first-order estimate $-2(\beta-1)$. Larger target changes expose the approximation error.

In [ ]:
import sys
import numpy as np
import matplotlib

if sys.platform != "emscripten":
    try:
        matplotlib.use("widget", force=True)
    except (RuntimeError, ValueError):
        from matplotlib.backends import backend_registry
        backend_registry._clear()
        matplotlib.use("widget", force=True)
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider

BLUE, TEAL, ORANGE, PURPLE, GRAY = "#2878b5", "#168578", "#e78b24", "#8854a5", "#707070"
plt.rcParams.update({"font.size": 11, "axes.titlesize": 11})

# Fixed observations and the physical response.
observed_errors = np.array([-2., -1., -1., 0.])  # min
observation_times = np.arange(4.)  # h

def evaluate_clock(decision):
    correction, rate = decision
    errors = observed_errors + correction + rate * observation_times
    gradient = 2 * np.array([errors.sum(), observation_times @ errors])
    return {"decision": np.array(decision), "errors": errors,
            "objective": float(errors @ errors), "gradient": gradient}

def solve_target(target):
    decision = ((8*target+1)/6, -(2*target+1)/6)
    result = evaluate_clock(decision)
    result["multiplier"] = (26-20*target)/3
    return result

In [ ]:
figure, axis = plt.subplots(figsize=(5.6, 4.8))
figure.subplots_adjust(left=0.15, right=0.96, top=0.90, bottom=0.40)
targets = np.linspace(0.7, 1.6, 200)
axis.plot(targets, 0.2 + (10/3)*(targets-1.3)**2, color=BLUE)
tangent, = axis.plot([], [], "--", color=TEAL)
candidate, = axis.plot([], [], "o", color=ORANGE)
axis.set(xlabel="Target beta (min)", ylabel="Best score V (min²)",
         xlim=(0.7, 1.6), ylim=(0, 1.5), title="Multiplier equals minus the tangent slope")
axis.grid(alpha=0.25)
status = figure.text(0.15, 0.24, "", va="top")
target_slider = Slider(figure.add_axes([0.25, 0.06, 0.58, 0.035]), "beta (min)",
                       0.7, 1.6, valinit=1, valstep=0.01, color=TEAL)

def update(_):
    target = target_slider.val
    result = solve_target(target)
    score, nu = result["objective"], result["multiplier"]
    q, r = result["decision"]
    candidate.set_data([target], [score])
    tangent.set_data(targets, score - nu*(targets-target))
    status.set_text(f"q = {q:.3f}; r = {r:.3f}; nu = {nu:.3f}\n"
                    f"Exact change from beta=1: {score-0.5:+.4f} min²\n"
                    f"Linear estimate from beta=1: {-2*(target-1):+.4f} min²")
    figure.canvas.draw_idle()

target_slider.on_changed(update)
figure._sliders = [target_slider]
update(None)
plt.show()